# Anomaly detection on wind turbine blade crops

A small convolutional autoencoder learns to reconstruct **healthy** blade
surface. Defects are found at test time because they reconstruct poorly.

**No defect labels are used for training.** The model only learns what *normal*
looks like; the defect boxes are used solely to locate evaluation crops, never
to train the model.

Steps:
1. Extract healthy crops (normal surface) and a held-out mix of healthy +
   defective crops.
2. Train the autoencoder on the healthy crops only.
3. Score every evaluation crop by its reconstruction error.
4. Compare healthy vs defective error distributions and render error heatmaps.
5. Measure ROC-AUC and AUPRC for the separation.

A T4 GPU trains this in a few minutes.

## 1. Environment

PyTorch, OpenCV, scikit-learn, matplotlib and NumPy are preinstalled on Colab.
The next cell confirms the GPU is visible.

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 2. Mount Drive and locate the dataset

The prepared WTBD dataset (output of `src/prepare_data.py`) is read from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATASET_DIR = Path('/content/drive/MyDrive/wtbd_yolo')   # dataset location on Drive
TRAIN_IMAGES = DATASET_DIR / 'images' / 'train'
TRAIN_LABELS = DATASET_DIR / 'labels' / 'train'
TEST_IMAGES = DATASET_DIR / 'images' / 'test'
TEST_LABELS = DATASET_DIR / 'labels' / 'test'
RESULTS_DIR = DATASET_DIR / 'results' / 'anomaly'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CROPS_DIR = Path('/content/anomaly_crops')               # extracted crops, local and fast
CROP, EPOCHS, BATCH, LR, BASE = 128, 20, 64, 1e-3, 16
PER_IMAGE, MAX_TRAIN, MAX_EVAL, SEED = 5, 4000, 600, 42

assert DATASET_DIR.exists(), f'Dataset not found at {DATASET_DIR}'
print('Dataset:', DATASET_DIR)

## 3. Extract crops

Healthy crops are square windows that do not overlap any defect box — ordinary
blade surface. Defective crops are square windows centred on each labelled
defect. Healthy training crops come from the train split; the evaluation healthy
and defective crops come from the held-out test split.

In [ ]:
import numpy as np, cv2, json
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 15, 'axes.titlesize': 18, 'axes.titleweight': 'bold',
    'axes.labelsize': 16, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 13,
})
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

def list_images(d):
    return sorted([p for e in IMAGE_EXTS for p in Path(d).rglob('*' + e)], key=lambda p: p.stem)

def load_boxes(label_path, w, h):
    if not label_path.exists():
        return np.zeros((0, 4), np.float32)
    rows = []
    for line in label_path.read_text().splitlines():
        s = line.split()
        if len(s) < 5:
            continue
        _, xc, yc, bw, bh = (float(v) for v in s[:5])
        rows.append([(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h])
    return np.array(rows, np.float32) if rows else np.zeros((0, 4), np.float32)

def window_overlaps(win, boxes, max_frac=0.0):
    if len(boxes) == 0:
        return False
    ix1 = np.maximum(win[0], boxes[:, 0]); iy1 = np.maximum(win[1], boxes[:, 1])
    ix2 = np.minimum(win[2], boxes[:, 2]); iy2 = np.minimum(win[3], boxes[:, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = (win[2] - win[0]) * (win[3] - win[1])
    return bool(np.any(inter / max(area, 1e-9) > max_frac))

def extract_healthy(images, labels_dir, out_dir, crop, per_image, max_crops, rng):
    out_dir.mkdir(parents=True, exist_ok=True); saved = 0
    for ip in images:
        if saved >= max_crops:
            break
        img = cv2.imread(str(ip))
        if img is None:
            continue
        h, w = img.shape[:2]
        boxes = load_boxes(labels_dir / (ip.stem + '.txt'), w, h)
        if min(h, w) < crop:
            sc = crop / min(h, w)
            img = cv2.resize(img, (max(int(round(w * sc)), crop), max(int(round(h * sc)), crop)))
            boxes = boxes * sc; h, w = img.shape[:2]
        for _ in range(per_image):
            if saved >= max_crops:
                break
            for _t in range(25):
                x = int(rng.integers(0, w - crop + 1)); y = int(rng.integers(0, h - crop + 1))
                if not window_overlaps(np.array([x, y, x + crop, y + crop], np.float32), boxes):
                    cv2.imwrite(str(out_dir / f'{ip.stem}_{saved:05d}.jpg'), img[y:y+crop, x:x+crop]); saved += 1
                    break
    return saved

def extract_defect(images, labels_dir, out_dir, crop, max_crops, margin=0.25):
    out_dir.mkdir(parents=True, exist_ok=True); saved = 0
    for ip in images:
        if saved >= max_crops:
            break
        img = cv2.imread(str(ip))
        if img is None:
            continue
        h, w = img.shape[:2]
        for box in load_boxes(labels_dir / (ip.stem + '.txt'), w, h):
            if saved >= max_crops:
                break
            bw, bh = box[2] - box[0], box[3] - box[1]
            side = float(min(max(max(bw, bh) * (1 + margin), crop * 0.5), min(h, w)))
            cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
            x1 = int(np.clip(cx - side / 2, 0, w - side)); y1 = int(np.clip(cy - side / 2, 0, h - side))
            patch = img[y1:y1 + int(side), x1:x1 + int(side)]
            if patch.size == 0:
                continue
            cv2.imwrite(str(out_dir / f'{ip.stem}_{saved:05d}.jpg'), cv2.resize(patch, (crop, crop))); saved += 1
    return saved

class CropDataset(Dataset):
    def __init__(self, crop_dir, crop):
        self.paths = list_images(crop_dir); self.crop = crop
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = cv2.cvtColor(cv2.imread(str(self.paths[i])), cv2.COLOR_BGR2RGB)
        if img.shape[:2] != (self.crop, self.crop):
            img = cv2.resize(img, (self.crop, self.crop))
        return torch.from_numpy(img).float().permute(2, 0, 1) / 255.0

print('Crop helpers ready.')

In [ ]:
rng = np.random.default_rng(SEED)
d_train = CROPS_DIR / 'train_healthy'
d_eval_healthy = CROPS_DIR / 'test_healthy'
d_eval_defect = CROPS_DIR / 'test_defect'

n_train = extract_healthy(list_images(TRAIN_IMAGES), TRAIN_LABELS, d_train, CROP, PER_IMAGE, MAX_TRAIN, rng)
n_eh = extract_healthy(list_images(TEST_IMAGES), TEST_LABELS, d_eval_healthy, CROP, PER_IMAGE, MAX_EVAL, rng)
n_ed = extract_defect(list_images(TEST_IMAGES), TEST_LABELS, d_eval_defect, CROP, MAX_EVAL)
print(f'healthy training crops: {n_train}')
print(f'eval healthy crops:     {n_eh}')
print(f'eval defective crops:   {n_ed}')

## 4. Autoencoder trained on healthy crops only

The network compresses a crop through four downsampling convolutions and
reconstructs it through four upsampling layers, optimising pixel-wise mean
squared error. It is shown healthy crops only, so it reproduces normal surface
well and unfamiliar defect texture poorly.

In [ ]:
import torch.nn as nn

class ConvAutoencoder(nn.Module):
    def __init__(self, channels=3, base=16):
        super().__init__()
        c1, c2, c3, c4 = base, base * 2, base * 4, base * 4
        def down(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 4, 2, 1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        def up(cin, cout, last=False):
            layers = [nn.ConvTranspose2d(cin, cout, 4, 2, 1)]
            layers += [nn.Sigmoid()] if last else [nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
            return nn.Sequential(*layers)
        self.encoder = nn.Sequential(down(channels, c1), down(c1, c2), down(c2, c3), down(c3, c4))
        self.decoder = nn.Sequential(up(c4, c3), up(c3, c2), up(c2, c1), up(c1, channels, last=True))
    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_autoencoder(model, loader, epochs, lr, device):
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr); crit = nn.MSELoss(); history = []
    for epoch in range(1, epochs + 1):
        running, n = 0.0, 0
        for batch in loader:
            batch = batch.to(device)
            opt.zero_grad(); loss = crit(model(batch), batch); loss.backward(); opt.step()
            running += loss.item() * batch.size(0); n += batch.size(0)
        history.append(running / max(n, 1))
        print(f'  epoch {epoch:>3}/{epochs}  mse={history[-1]:.5f}')
    return history

torch.manual_seed(SEED)
train_set = CropDataset(d_train, CROP)
loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, drop_last=len(train_set) >= BATCH)
model = ConvAutoencoder(channels=3, base=BASE)
print(f'Training on {len(train_set)} healthy crops for {EPOCHS} epochs:')
history = train_autoencoder(model, loader, EPOCHS, LR, device)
torch.save(model.state_dict(), RESULTS_DIR / 'autoencoder.pt')
print('Saved ->', RESULTS_DIR / 'autoencoder.pt')

## 5. Reconstruction error on healthy vs defective crops

Each evaluation crop is scored by its mean squared reconstruction error. Healthy
crops reconstruct well (low error); defective crops reconstruct poorly (high
error).

In [ ]:
@torch.no_grad()
def reconstruction_errors(model, dataset, device, batch_size=64):
    model.to(device).eval(); errs = []
    for batch in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        batch = batch.to(device)
        errs.append(((batch - model(batch)) ** 2).mean(dim=[1, 2, 3]).cpu().numpy())
    return np.concatenate(errs) if errs else np.zeros((0,), np.float32)

@torch.no_grad()
def error_map(model, crop_tensor, device):
    model.to(device).eval()
    x = crop_tensor.unsqueeze(0).to(device); recon = model(x)
    err = ((x - recon) ** 2).mean(dim=1)[0].cpu().numpy()
    return recon[0].cpu().numpy().transpose(1, 2, 0), err

eval_healthy = CropDataset(d_eval_healthy, CROP)
eval_defect = CropDataset(d_eval_defect, CROP)
err_healthy = reconstruction_errors(model, eval_healthy, device, BATCH)
err_defect = reconstruction_errors(model, eval_defect, device, BATCH)
print(f'mean error  healthy = {err_healthy.mean():.5f}')
print(f'mean error  defect  = {err_defect.mean():.5f}')
print(f'ratio defect/healthy = {err_defect.mean() / max(err_healthy.mean(), 1e-9):.1f}x')

## 6. Error distributions

The two distributions separate: defective crops sit at higher reconstruction
error. The dashed line marks the 95th percentile of healthy error, a candidate
alarm threshold.

In [ ]:
threshold = float(np.percentile(err_healthy, 95))
fig, ax = plt.subplots(figsize=(10, 7))
bins = np.linspace(min(err_healthy.min(), err_defect.min()), max(err_healthy.max(), err_defect.max()), 40)
ax.hist(err_healthy, bins=bins, density=True, alpha=0.6, color='#2c7fb8', label=f'healthy (n={len(err_healthy)})')
ax.hist(err_defect, bins=bins, density=True, alpha=0.6, color='#d95f0e', label=f'defective (n={len(err_defect)})')
ax.axvline(threshold, color='black', linestyle='--', lw=2, label=f'healthy 95th pct = {threshold:.4f}')
ax.set_xlabel('Reconstruction error (mean squared error)'); ax.set_ylabel('Density')
ax.set_title('Reconstruction error: healthy vs defective crops')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(RESULTS_DIR / 'error_distributions.png', bbox_inches='tight', dpi=150); plt.show()
print('Saved ->', RESULTS_DIR / 'error_distributions.png')

## 7. Reconstruction-error heatmaps

For the highest-error defective crops, the per-pixel error localises the defect:
the bright region of the heatmap marks where the surface departs from normal.

In [ ]:
order = np.argsort(-err_defect)[:4]
fig, axes = plt.subplots(len(order), 3, figsize=(12, 4 * len(order)))
if len(order) == 1:
    axes = axes[None, :]
for row, idx in enumerate(order):
    x = eval_defect[idx]; recon, err = error_map(model, x, device); inp = x.numpy().transpose(1, 2, 0)
    axes[row, 0].imshow(np.clip(inp, 0, 1)); axes[row, 0].set_title('Input' if row == 0 else '')
    axes[row, 1].imshow(np.clip(recon, 0, 1)); axes[row, 1].set_title('Reconstruction' if row == 0 else '')
    axes[row, 2].imshow(np.clip(inp, 0, 1))
    hm = axes[row, 2].imshow(err, cmap='jet', alpha=0.55)
    axes[row, 2].set_title('Error heatmap (defect highlighted)' if row == 0 else '')
    fig.colorbar(hm, ax=axes[row, 2], fraction=0.046, pad=0.04)
    for col in range(3):
        axes[row, col].axis('off')
fig.suptitle('Reconstruction-error heatmaps on defective crops', fontsize=18, fontweight='bold')
fig.savefig(RESULTS_DIR / 'heatmaps.png', bbox_inches='tight', dpi=150); plt.show()
print('Saved ->', RESULTS_DIR / 'heatmaps.png')

## 8. Separation score: ROC-AUC and AUPRC

Treating defective as the positive class and reconstruction error as the score,
ROC-AUC and AUPRC summarise how well the error separates defective from healthy
crops — with no defect labels involved in training.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

labels = np.concatenate([np.zeros(len(err_healthy)), np.ones(len(err_defect))])
scores = np.concatenate([err_healthy, err_defect])
auc = float(roc_auc_score(labels, scores)); ap = float(average_precision_score(labels, scores))
detect_rate = float((err_defect > threshold).mean()); false_pos_rate = float((err_healthy > threshold).mean())

fpr, tpr, _ = roc_curve(labels, scores); prec, rec, _ = precision_recall_curve(labels, scores)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].plot(fpr, tpr, lw=3, color='#2c7fb8'); axes[0].plot([0, 1], [0, 1], '--', color='grey', lw=1.5)
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title(f'ROC  (AUC = {auc:.3f})'); axes[0].grid(True, alpha=0.3)
axes[1].plot(rec, prec, lw=3, color='#d95f0e')
axes[1].set_xlabel('Recall (defects caught)'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall  (AUPRC = {ap:.3f})'); axes[1].grid(True, alpha=0.3)
fig.suptitle('Healthy-vs-defect separation by reconstruction error', fontsize=18, fontweight='bold')
fig.savefig(RESULTS_DIR / 'roc_pr_curves.png', bbox_inches='tight', dpi=150); plt.show()

metrics = {
    'crop_size': CROP, 'epochs': EPOCHS, 'final_train_mse': round(history[-1], 6),
    'counts': {'train_healthy': len(train_set), 'test_healthy': len(eval_healthy), 'test_defect': len(eval_defect)},
    'reconstruction_error': {
        'healthy_mean': round(float(err_healthy.mean()), 6), 'healthy_std': round(float(err_healthy.std()), 6),
        'defect_mean': round(float(err_defect.mean()), 6), 'defect_std': round(float(err_defect.std()), 6)},
    'separation': {'roc_auc': round(auc, 4), 'auprc': round(ap, 4)},
    'operating_point': {'threshold_healthy_p95': round(threshold, 6),
                        'defect_detection_rate': round(detect_rate, 4),
                        'healthy_false_positive_rate': round(false_pos_rate, 4)},
    'note': 'Trained on healthy crops only; defect labels were not used in training.',
}
(RESULTS_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'ROC-AUC = {auc:.3f}    AUPRC = {ap:.3f}')
print(f'at the healthy 95th-pct threshold: {detect_rate:.1%} of defects flagged, '
      f'{false_pos_rate:.1%} healthy false alarms')
print('Saved ->', RESULTS_DIR / 'metrics.json')

## Done

Saved to `wtbd_yolo/results/anomaly/`:

- `autoencoder.pt` — trained weights.
- `metrics.json` — counts, error statistics, ROC-AUC, AUPRC, operating point.
- `error_distributions.png`, `roc_pr_curves.png`, `heatmaps.png` — figures.

The detector was built from healthy crops alone, so it extends to defect types
that were rare or unlabelled in the detection data.